# 06 — Pull and normalize skills

Reuse the existing extraction and dictionary pipeline; no duplicate extractor is implemented. Raw punctuation is preserved for C++, C#, .NET and 1С. Canonical skill evidence is ID-aligned, and missing evidence means unknown. Dictionary v2 remains agent-reviewed with human review pending.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'recommender').is_dir()), None)
assert PROJECT is not None, 'Open within the project folder'
import os
RESULTS = Path(os.environ.get('RESULTS_DIR', str(PROJECT / 'results/final'))).resolve()
import sys
sys.path.insert(0, str(PROJECT))
from m9_skills.extraction import Extractor
from m9_skills.workflow import extract
from m9_skills.compatibility import align_sidecar
DICTIONARY = PROJECT / 'm9_skills/dictionary.v2.json'
RAW = PROJECT / 'datasets/HSE_dataset(D4).csv'
SIDECARS = PROJECT / 'm9_skills/outputs/v2_validation_run'
engine = Extractor(DICTIONARY)


In [2]:
# Existing completed extraction is reused; choose a fresh output folder to regenerate.
assert (SIDECARS / 'coverage.json').exists(), 'Run the pipeline first.'
coverage = json.loads((SIDECARS / 'coverage.json').read_text())
assert coverage['dictionary_sha256'] == engine.fingerprint
display({k: coverage[k] for k in ['dictionary_version', 'human_review_complete', 'coverage_is_accuracy']})
display(pd.DataFrame({side: {'entities': d['entities'], 'mention_coverage': d['mention_coverage']} for side, d in coverage['diagnostics'].items()}))


{'dictionary_version': '0.2.0-agent-reviewed-opt-in',
 'human_review_complete': False,
 'coverage_is_accuracy': False}

,cv,vacancy
entities,65714.000000,43756.000000
mention_coverage,0.339288,0.376794


In [3]:
example = 'Знание Excel и 1С; Python не требуется.'
display(pd.DataFrame(engine.extract(example, field='positionRequirements')))


,skill_id,canonical,alias,matched,start,end,field,source,assertion,requirement,esco
0,excel,Microsoft Excel,excel,Excel,7,12,positionRequirements,phrase_rule,mentioned,required,None
1,1c,1С,1с,1С,15,17,positionRequirements,phrase_rule,mentioned,unspecified,None
2,python,Python,python,Python,19,25,positionRequirements,phrase_rule,not_required,not_required,None


## Existing linguistic validation

Coverage is not extraction completeness or recommendation accuracy. The corpus diagnostic has 39 scored cases; the authored challenge has 40. Agent annotations are not completed human validation. Known coordinated-negation and aspiration errors remain visible. The scoring policy excludes soft traits and resume title/education-only evidence.


In [4]:
diagnostics = {}
for name in ['baseline', 'expanded']:
    report = json.loads((PROJECT / f'm9_skills/validation/results/{name}.json').read_text())
    diagnostics[name] = report['by_suite']
display(diagnostics)


{'baseline': {'authored_challenge': {'cases': 40,
   'true_positive': 13,
   'false_positive': 3,
   'false_negative': 21,
   'precision': 0.8125,
   'recall': 0.38235294117647056,
   'f1': 0.52,
   'exact_match_cases': 16,
   'assertion_error_cases': 3,
   'cases_with_out_of_inventory_gaps': 0},
  'corpus_diagnostic': {'cases': 39,
   'true_positive': 6,
   'false_positive': 0,
   'false_negative': 7,
   'precision': 1.0,
   'recall': 0.46153846153846156,
   'f1': 0.631578947368421,
   'exact_match_cases': 32,
   'assertion_error_cases': 0,
   'cases_with_out_of_inventory_gaps': 26}},
 'expanded': {'authored_challenge': {'cases': 40,
   'true_positive': 32,
   'false_positive': 3,
   'false_negative': 2,
   'precision': 0.9142857142857143,
   'recall': 0.9411764705882353,
   'f1': 0.927536231884058,
   'exact_match_cases': 35,
   'assertion_error_cases': 3,
   'cases_with_out_of_inventory_gaps': 0},
  'corpus_diagnostic': {'cases': 39,
   'true_positive': 8,
   'false_positive': 0,
  

## Downstream contract

`recommender.experiment` calls `align_sidecar` with exact ordered applicant/vacancy IDs and the dictionary hash. Never attach by row position. First raw snapshots differ from cleaned M8 snapshots; model results assess the complete policy, not skill extraction in isolation. Read notebook 07 after the pipeline finishes.
